In [3]:
import os
import random
import cv2
import numpy as np
from ultralytics import YOLO

# --- CONFIG ---
input_dir = "E:/Datasets/Dataset_B/OLD/Dataset_B-ORIGINAL"          # directory of images to mask
background_dir = "E:/Datasets/CocoExperiment/COCO_W_KAAKAA_dataset/full_org_dataset/images/train" # directory of COCO images
output_dir = "E:/Datasets/Dataset_B/composite_kaakaa_coco"    # save results here
os.makedirs(output_dir, exist_ok=True)

# Load YOLO segmentation model
model = YOLO("C:/Users/alexw/Assignments/AIML339/Project/instance_segmentation/BEST_MODEL_YOLO/best-kaakaa-yolo11n-seg.pt")  # or your custom weights


def load_random_background(target_shape):
    """Pick a random background image from COCO and resize to match target shape (h, w)."""
    bg_file = random.choice(os.listdir(background_dir))
    bg = cv2.imread(os.path.join(background_dir, bg_file))
    bg = cv2.resize(bg, (target_shape[1], target_shape[0]))  # (W, H)
    return bg

def overlay_masked_on_background(fg, mask, bg):
    """Overlay foreground object (fg) masked onto background bg (all same size)."""
    mask_3c = np.stack([mask]*3, axis=-1)  # expand mask to 3 channels
    fg_masked = fg * mask_3c
    bg_masked = bg * (1 - mask_3c)
    return fg_masked + bg_masked

# Walk through subdirectories
for root, _, files in os.walk(input_dir):
    for fname in files:
        if not fname.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        # Paths
        img_path = os.path.join(root, fname)
        rel_path = os.path.relpath(root, input_dir)  # preserve subfolder structure
        out_dir = os.path.join(output_dir, rel_path)
        os.makedirs(out_dir, exist_ok=True)

        # Load image
        img = cv2.imread(img_path)
        results = model(img, verbose=False)

        saved_any = False

        for r in results:
            masks = r.masks
            boxes = r.boxes

            if masks is None or boxes is None:
                continue

            for j, (mask_tensor, box) in enumerate(zip(masks.data, boxes.xyxy)):
                # Convert mask to numpy
                mask = mask_tensor.cpu().numpy().astype(np.uint8)  # (H, W)
                mask = cv2.resize(mask, (img.shape[1], img.shape[0]))  # resize mask to image size

                # Get bounding box
                x1, y1, x2, y2 = map(int, box.tolist())
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)

                # Crop image + mask to bbox
                fg_crop = img[y1:y2, x1:x2]
                mask_crop = mask[y1:y2, x1:x2]

                if fg_crop.size == 0 or mask_crop.size == 0:
                    continue

                # Get random background same size as bbox
                bg = load_random_background(fg_crop.shape[:2])

                # Composite
                composite = overlay_masked_on_background(fg_crop, mask_crop, bg)

                # Save with same name
                out_path = os.path.join(out_dir, f"{os.path.splitext(fname)[0]}_inst{j}.png")
                cv2.imwrite(out_path, composite)
                saved_any = True

        if not saved_any:
            # If nothing detected, optionally copy original to output
            out_path = os.path.join(out_dir, fname)
            cv2.imwrite(out_path, img)

print("Done! Cropped composites saved in:", output_dir)

Done! Cropped composites saved in: E:/Datasets/Dataset_B/composite_kaakaa_coco
